# Peek at KO/EC links — annotation rows ↔ biosamples / studies

Worked-example queries showing how to navigate from a row in
`nmdc_results.annotation_kegg_orthology` (or `annotation_enzyme_commission`)
out to the biosample, study, and file metadata in `nmdc_metadata`.

**Anchor columns on the annotation tables:**

| column | meaning | join target |
|---|---|---|
| `workflow_run_id` | `nmdc:wfmgan-...` MetagenomeAnnotation activity | `nmdc_metadata.workflow_execution_set.id` |
| `data_object_id`  | `nmdc:dobj-...` source TSV file | `nmdc_metadata.data_object_set.id` |
| `gene_id`         | `<workflow_run_id>_<contig>_<start>_<end>` | local; matches functional GFFs (issues #84, #86–#90) |
| `ncbi_taxid`      | NCBI taxon for the BLAST hit | external (no native BERDL taxonomy table) |
| `annotation_id`   | `KO:Kxxxxx` or `EC:n.n.n.n` | KEGG / EC ontologies |

**The chain:**

```
annotation_*.workflow_run_id
  → workflow_execution_set.id  (.was_informed_by is array of dgns IDs)
  → data_generation_set.id
    → data_generation_set_has_input.has_input            (biosample IDs)
    → data_generation_set_associated_studies.associated_studies
  → biosample_set.{env_*, geo_loc_name_*, depth, …}
```

### Session setup

In [1]:
spark = get_spark_session(app_name="peek_ko_ec_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

### Preflight: confirm the annotation tables are registered

In [2]:
for tbl in ("annotation_kegg_orthology", "annotation_enzyme_commission"):
    if spark.sql(f"SHOW TABLES IN nmdc_results LIKE '{tbl}'").count() == 0:
        raise RuntimeError(f"nmdc_results.{tbl} not registered — run ingest_ko_ec_annotations.ipynb first")
    print(f"OK: nmdc_results.{tbl}")

OK: nmdc_results.annotation_kegg_orthology
OK: nmdc_results.annotation_enzyme_commission


### 1. Pick an anchor row

Grab one annotation row to drive the rest of the joins. Same recipe works for EC.

In [3]:
anchor = (
    spark.sql("""
        SELECT gene_id, annotation_id, workflow_run_id, data_object_id, ncbi_taxid
        FROM nmdc_results.annotation_kegg_orthology
        LIMIT 1
    """)
    .toPandas()
    .iloc[0]
)
WORKFLOW_RUN_ID = anchor["workflow_run_id"]
DATA_OBJECT_ID  = anchor["data_object_id"]
ANNOTATION_ID   = anchor["annotation_id"]
print(anchor.to_string())

gene_id            nmdc:wfmgan-11-4b91dx77.1_2431468_82_447
annotation_id                                     KO:K03752
workflow_run_id                   nmdc:wfmgan-11-4b91dx77.1
data_object_id                        nmdc:dobj-11-r4jvp282
ncbi_taxid                                       2995780880


### 2. `workflow_run_id` → workflow_execution_set

`was_informed_by` is **array-typed** (a workflow run can be informed by
multiple data generations). Use `array_contains` or `EXPLODE` to join through
it — equality won't work.

In [4]:
spark.sql(f"""
    SELECT id, type, was_informed_by, started_at_time, ended_at_time
    FROM nmdc_metadata.workflow_execution_set
    WHERE id = '{WORKFLOW_RUN_ID}'
""").toPandas()

,id,type,was_informed_by,started_at_time,ended_at_time
0,nmdc:wfmgan-11-4b91dx77.1,nmdc:MetagenomeAnnotation,[nmdc:dgns-11-e12jgq19],2025-07-14 20:38:51,2025-07-26 20:48:43


### 3. workflow_execution → data_generation_set (the sequencing run)

In [5]:
spark.sql(f"""
    SELECT dg.id, dg.type, dg.name, dg.processing_institution,
           dg.principal_investigator_name, dg.start_date, dg.end_date
    FROM (
        SELECT EXPLODE(was_informed_by) AS dgns_id
        FROM nmdc_metadata.workflow_execution_set
        WHERE id = '{WORKFLOW_RUN_ID}'
    ) info
    JOIN nmdc_metadata.data_generation_set dg
      ON dg.id = info.dgns_id
""").toPandas()


,id,type,name,processing_institution,principal_investigator_name,start_date,end_date
0,nmdc:dgns-11-e12jgq19,nmdc:NucleotideSequencing,Forest soil microbial communities from Barre W...,JGI,Jeffrey Blanchard,NaN,NaN


### Aside: provenance graph and `biosample_to_annotation_run`

NMDC's data model has two parallel chains, bridged by `data_generation`:

```
biosample → material_processing → procsm → … → data_generation
                                                       ↓
                              data_object ← workflow_execution
```

Walking this graph from a workflow run back to source biosamples was previously
done with a Trino `WITH RECURSIVE` CTE (which hits a 150-stage planner limit
at scale) or an iterative Python BFS. Both are now replaced by a single JOIN
to the precomputed `nmdc_metadata.biosample_to_annotation_run` table.

That table was built by `notebooks/build_biosample_to_annotation_run.ipynb`
and must be rebuilt after each NMDC data load.

### 4. workflow_execution → biosample(s)

A direct JOIN to `biosample_to_annotation_run` replaces the recursive graph
walk. The precomputed table covers any chain depth and records which
MaterialProcessing classes appeared in the provenance chain.

In [7]:
spark.sql(f"""
    SELECT b2ar.biosample_id, b2ar.n_hops,
           bsm.env_broad_scale_term_id, bsm.env_local_scale_term_id,
           bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value,
           bsm.depth_has_numeric_value, bsm.depth_has_unit
    FROM   nmdc_metadata.biosample_to_annotation_run b2ar
    JOIN   nmdc_metadata.biosample_set bsm ON bsm.id = b2ar.biosample_id
    WHERE  b2ar.workflow_run_id = '{WORKFLOW_RUN_ID}'
""").toPandas()

### 5. data_generation → study

In [8]:
spark.sql(f"""
    SELECT s.id AS study_id, s.name, s.title, s.principal_investigator_name
    FROM (
        SELECT EXPLODE(was_informed_by) AS dgns_id
        FROM nmdc_metadata.workflow_execution_set
        WHERE id = '{WORKFLOW_RUN_ID}'
    ) info
    JOIN nmdc_metadata.data_generation_set_associated_studies dgs
      ON dgs.parent_id = info.dgns_id
    JOIN nmdc_metadata.study_set s
      ON s.id = dgs.associated_studies
""").toPandas()


,study_id,name,title,principal_investigator_name
0,nmdc:sty-11-8ws97026,Molecular mechanisms underlying changes in the...,Molecular mechanisms underlying changes in the...,Jeffrey Blanchard


### 6. `data_object_id` → data_object_set (file-level provenance)

Useful for verifying which source TSV a row came from, file size, MD5,
URL, etc. Not needed for downstream science.

In [9]:
spark.sql(f"""
    SELECT id, name, data_object_type, file_size_bytes, md5_checksum, url
    FROM nmdc_metadata.data_object_set
    WHERE id = '{DATA_OBJECT_ID}'
""").toPandas()

,id,name,data_object_type,file_size_bytes,md5_checksum,url
0,nmdc:dobj-11-r4jvp282,nmdc_wfmgan-11-4b91dx77.1_ko.tsv,Annotation KEGG Orthology,249951168,42106ee20e0a956059025853ea50ddc8,https://data.microbiomedata.org/data/nmdc:dgns...


### 7. Cross-check against `functional_annotation_agg`

`functional_annotation_agg` is a precomputed `(workflow_run, gene_function_id, count)`
rollup. Two caveats when joining to it:

1. **EC is absent** — `functional_annotation_agg` only carries `PFAM`,
   `KEGG.ORTHOLOGY`, and `COG`. The new `annotation_enzyme_commission` table
   is the only source of EC in BERDL.
2. **KO prefix differs** — `KO:Kxxxxx` here vs `KEGG.ORTHOLOGY:Kxxxxx` there.
   Translate before joining. The two values per workflow run *should* line up
   (`COUNT(*)` in `annotation_kegg_orthology` ≈ `SUM(count)` in the agg);
   any drift is worth flagging.

In [10]:
spark.sql(f"""
    WITH ours AS (
        SELECT 'KEGG.ORTHOLOGY:' || SUBSTRING(annotation_id, 4) AS gene_function_id,
               COUNT(*) AS hits_in_ko_table
        FROM nmdc_results.annotation_kegg_orthology
        WHERE workflow_run_id = '{WORKFLOW_RUN_ID}'
        GROUP BY 1
    ),
    theirs AS (
        SELECT gene_function_id, count AS count_in_agg
        FROM nmdc_metadata.functional_annotation_agg
        WHERE was_generated_by = '{WORKFLOW_RUN_ID}'
          AND gene_function_id LIKE 'KEGG.ORTHOLOGY:%'
    )
    SELECT COALESCE(ours.gene_function_id, theirs.gene_function_id) AS gene_function_id,
           ours.hits_in_ko_table,
           theirs.count_in_agg,
           ours.hits_in_ko_table - theirs.count_in_agg AS diff
    FROM ours FULL OUTER JOIN theirs USING (gene_function_id)
    ORDER BY ABS(COALESCE(ours.hits_in_ko_table, 0) - COALESCE(theirs.count_in_agg, 0)) DESC
    LIMIT 20
""").toPandas()

,gene_function_id,hits_in_ko_table,count_in_agg,diff
0,KEGG.ORTHOLOGY:K00001,559,559,0
1,KEGG.ORTHOLOGY:K00002,149,149,0
2,KEGG.ORTHOLOGY:K00003,1196,1196,0
3,KEGG.ORTHOLOGY:K00004,191,191,0
4,KEGG.ORTHOLOGY:K00005,28,28,0
5,KEGG.ORTHOLOGY:K00007,6,6,0
6,KEGG.ORTHOLOGY:K00008,1515,1515,0
7,KEGG.ORTHOLOGY:K00009,7,7,0
8,KEGG.ORTHOLOGY:K00010,900,900,0
9,KEGG.ORTHOLOGY:K00012,2043,2043,0


### 8. End-to-end: KO hits per biosample for a chosen KO

A single Spark SQL query joins the annotation table to
`biosample_to_annotation_run` — no recursive walk, no Trino cursor.
The precomputed provenance bridge does the graph traversal for free.

In [11]:
import time
TARGET_KO = ANNOTATION_ID  # substitute any 'KO:Kxxxxx'

t0 = time.monotonic()
result = spark.sql(f"""
    SELECT b2ar.biosample_id,
           bsm.env_broad_scale_term_id,
           bsm.env_medium_term_id,
           bsm.geo_loc_name_has_raw_value,
           SUM(ko.n_hits) AS total_hits
    FROM (
        SELECT workflow_run_id, COUNT(*) AS n_hits
        FROM   nmdc_results.annotation_kegg_orthology
        WHERE  annotation_id = '{TARGET_KO}'
        GROUP  BY workflow_run_id
    ) ko
    JOIN nmdc_metadata.biosample_to_annotation_run b2ar
      ON b2ar.workflow_run_id = ko.workflow_run_id
    JOIN nmdc_metadata.biosample_set bsm
      ON bsm.id = b2ar.biosample_id
    GROUP BY b2ar.biosample_id, bsm.env_broad_scale_term_id,
             bsm.env_medium_term_id, bsm.geo_loc_name_has_raw_value
    ORDER BY total_hits DESC
    LIMIT 20
""").toPandas()
print(f"{len(result)} biosamples  ({time.monotonic()-t0:.1f}s)")
result